# 학습데이터 증강&평가

### 계약서 증강


In [1]:
!pip install -q openai tqdm

In [2]:
import json, glob, os, re, time, random, os
from google.colab import userdata
from openai import OpenAI
from getpass import getpass
from openai import OpenAI
from tqdm import tqdm

API_KEY = userdata.get('OPENAI_API_KEY')
client = OpenAI(api_key=API_KEY)
MODEL = "gpt-4o"

In [4]:
# 처음에만 돌려도 됨 -> 이미 train/valid 다 나눔
import json, random
from collections import defaultdict, Counter

raw = json.load(open('CON_seed_v3.json', encoding='utf-8'))   # ← final 아니라 v3(확정 골든)
data = raw['data'] if isinstance(raw, dict) and 'data' in raw else raw

by_label = defaultdict(list)
for r in data:
    by_label[r['label']].append(r)

random.seed(42)
train, valid = [], []
VALID_RATIO = 0.30   # label별 30%를 valid로 (검토 10→valid 3, train 7 식)

for label, items in by_label.items():
    random.shuffle(items)
    n_valid = max(1, round(len(items) * VALID_RATIO))
    valid += items[:n_valid]
    train += items[n_valid:]

random.shuffle(train); random.shuffle(valid)
print(f"train {len(train)} / valid {len(valid)}")
print("train 분포:", dict(Counter(r['label'] for r in train)))
print("valid 분포:", dict(Counter(r['label'] for r in valid)))

json.dump(train, open('CON_train.json', 'w'), ensure_ascii=False, indent=2)
json.dump(valid, open('CON_valid.json', 'w'), ensure_ascii=False, indent=2)
print("저장: CON_train.json, CON_valid.json")

FileNotFoundError: [Errno 2] No such file or directory: 'CON_seed_v3.json'

In [9]:
# 라벨별 증강 목표 (통과 후 최종 개수) — 지표 근거: 검토 R0.33, 불일치 R0.43 약함
CON_TARGET = {"검토": 28, "불일치": 14, "일치": 8}   # 합 +50
OVERGEN    = {"검토": 3.0, "불일치": 2.0, "일치": 2.0}  # 탈락 고려 초과생성 배수
N_PER_CALL      = 4       # 1회 호출당 생성 변형 수 (기존 N 역할)
DISTRACTOR_RATE = 0.3
TEMP            = 0.9

GOLDEN  = "CON_train_35.json"
JO_GLOB = "*_jo.json"

In [7]:
CON_AUG_PROMPT = """당신은 공공 SW 계약서 조항 검토 학습데이터를 증강하는 생성기다.
입력으로 검증된 골든 케이스 1건(JSON)을 받으면, 원본 판정을 유지하는 변형 {N}건을 JSON 배열로만 출력한다.

[절대 규칙]

1. output 역할의 핵심 필드는 원본과 동일하게 유지한다.

* label은 원본과 100% 동일해야 한다.
* pattern은 원본과 100% 동일하게 유지한다.
* category는 원본과 100% 동일하게 유지한다.
* refs는 원본의 배열을 그대로 복사한다. chunk_id, article, law_name, chunk_text를 추가·삭제·수정하지 않는다.

2. refs 밖 판단 금지

* clause_text와 refs만 사용한다.
* refs에 없는 법령명, 조문명, 항, 호, 수치, 기간, 예외, 요건을 외부 지식으로 보완하지 않는다.
* 사고과정과 근거에는 반드시 실제 참고한 refs의 law_name과 article을 드러낸다.

3. 법률결론어 금지

* 사고과정과 근거 본문에는 다음 표현을 쓰지 않는다.
  "위반", "부당", "무효", "적법", "위법", "소지", "정당"
* 허용되는 표현 예:
  "~와 동일함", "~보다 높게 책정됨", "~보다 낮게 책정됨", "~범위를 벗어남",
  "을이 부담하는 금액이 커짐", "계약상대자의 부담이 줄어듦",
  "발주기관이 확보하는 금액이 줄어듦", "대조 기준이 refs에 없음"

4. disclaimer 고정

* label이 "일치" 또는 "불일치"이면 근거는 반드시 다음 문장으로 끝난다.
  "(참고용 검토이며 법적 판단 아님)"
* label이 "검토"이면 근거는 반드시 다음 문장으로 끝난다.
  "(참고자료 부족에 따른 검토 보류이며 법적 판단 아님)"

5. 사고과정 형식

* 원본이 "초안: → 재검토: → 수정:" 형식이면 그대로 유지한다.
* 원본이 다른 형식이면 다음 흐름으로 작성한다.
  "① clause_text의 핵심 수치·조건 → ② 실제 참고 refs 법령·조문 → ③ 대조 → 결론"
* 사고과정에는 반드시 실제 참고한 법령명과 조문을 1회 이상 포함한다.
* 검토(RAG miss)인 경우에도 어떤 refs를 확인했는지 쓰고, 그 refs에 판단 기준이 없다고 명시한다.

6. 출력은 JSON 배열만 허용한다.
   설명문, 코드블록, 주석을 출력하지 않는다.

[핵심 판정 단위]

* clause_text 전체를 통째로 보지 말고, 판정에 실질적으로 영향을 주는 핵심 수치·조건을 먼저 식별한다.
* 핵심 수치·조건이 여러 개이면 각 항목을 따로 대조한 뒤 전체 label을 결정한다.
* 다항 worst-case에서는 문제를 일으키는 항 또는 문장을 사고과정과 근거에 명시한다.

[라벨 판정 기준 — 변형 시 이 경계를 반드시 보존]

1. 일치

* label은 "일치"다.
* clause_text의 핵심 수치·조건이 refs.chunk_text 기준과 동일하다.
* refs에 비교 가능한 기준이 실제로 존재해야 한다.

2. 일치경계(B)

* pattern은 "일치경계(B)"이지만 label은 반드시 "일치"다.
* 핵심 수치·조건은 refs 기준과 같다.
* 다만 refs가 세부조건, 기간, 예외, 기산 방식, 운영 절차까지는 명시하지 않아 일부가 더 엄격해 보이더라도 단정하기 어렵다.
* 핵심 비교 기준이 같으면 "검토"나 "불일치"로 바꾸지 않는다.
* 특히 refs가 원칙과 함께 예외·단서(재난 시 단축 등)를 규정하는데 clause_text가 예외를 생략하고 원칙만 규정한 경우, 원칙 수치가 refs와 같으면 label은 "일치"다. 예외 생략만을 이유로 불일치로 만들지 마라.

3. 불일치

* label은 "불일치"다.
* 핵심 수치·조건이 refs 기준과 다르다.
* 차이 유형은 초과, 미달, 배제, 면제, 생략, 의무 가중, 권리 제한, 순서 변경, 조건 삭제 등일 수 있다.
* 방향성은 사실 관찰로만 쓴다.
* 해당 category가 을불리/을유리로 자연스럽게 설명되면 그렇게 쓰고,
  그렇지 않으면 어느 당사자 또는 어떤 계약 효과가 달라지는지 사실로 쓴다.
* refs의 예외·단서를 clause가 생략한 것은 불일치가 아니다. 핵심 수치·조건의 '값 자체'가 refs와 어긋날 때만 불일치다.

4. 검토(RAG miss)

* label은 "검토"다.
* clause_text의 쟁점을 판단하는 decisive 기준이 refs.chunk_text에 없다.
* refs가 같은 주제처럼 보이더라도, 핵심 수치·조건을 직접 비교할 기준이 없으면 검토다.
* 외부 지식으로 원래의 정답 조문을 상정해 일치/불일치로 바꾸면 안 된다.

[판정별 보존 규칙 — 위반 시 케이스 폐기]

* 일치: 핵심 수치·조건의 동일 상태를 유지한다.
* 일치경계(B): 핵심 기준 동일 + 세부 영역 미명시 상태를 유지한다.
* 불일치: 원본의 어긋난 지점을 유지한다. 초과/미달/배제/면제/생략의 방향을 바꾸지 않는다.
* 검토: decisive 기준이 refs에 없는 상태를 유지한다. 관련 refs를 새로 암시해 결함을 해소하면 안 된다.

[변형 유형 — 골고루 사용]
A. 표현 변형

* 같은 수치·조건을 다른 어순, 다른 표기, 동의어로 바꾼다.
* 예: "1000분의 3" ↔ "1천분의 3", "검사 완료 후" ↔ "검사를 마친 뒤"

B. 다항 확장(worst-case)

* 조문을 ①②③ 여러 항으로 확장하되, 원본의 핵심 판정 원인을 그대로 유지한다.
* 불일치 원인이 한 항에만 있으면 사고과정과 근거에 해당 항을 명시한다.

C. 표면 노이즈

* 띄어쓰기 변형, 경미한 표현 흔들림, 무관 문장 1개 삽입은 허용한다.
* 단, 핵심 수치·조건은 바꾸지 않는다.
* 노이즈가 판정을 바꾸거나 결함을 해소하면 안 된다.

[시나리오 작성 규칙]

* 형식: "쟁점 요약 (변형유형)" 1줄
* 길이: 10~25자
* 원본의 쟁점 방향은 유지한다.
* 변형유형 태그 예:
  "(표현변형)", "(다항 worst-case)", "(노이즈)"

[사고과정·근거 작성 규칙]

* 사고과정에는 실제 참고한 법령명과 조문을 1회 이상 포함한다.
* 근거에도 실제 참고한 법령명과 조문을 1회 이상 포함한다.
* 여러 refs가 있어도 실제 판단에 사용한 refs만 언급한다.
* refs가 2개 이상이고 서로 다른 기준을 제공하면 근거에 필요한 refs를 모두 반영한다.
* "refs 기준"처럼 뭉뚱그려 쓰지 말고, "지방계약법 시행규칙 제75조", "지방계약법 시행령 제90조제3항"처럼 쓴다.
* refs metadata에 없는 항·호는 추정해 쓰지 않는다.
* refs.chunk_text에 없는 수치나 요건을 새로 만들어 넣지 않는다.

[출력 스키마]
[
{
"id": "{원본id}_aug1",
"category": "{원본과 동일}",
"pattern": "{원본과 동일}",
"시나리오": "...",
"clause_text": "...",
"refs": [{원본 refs 그대로}],
"사고과정": "...",
"label": "{원본과 동일}",
"근거": "..."
}
]

[최종 점검]
각 케이스를 출력하기 전 반드시 확인한다.

* label, pattern, category가 원본과 동일한가
* refs가 원본과 완전히 동일한가
* 사고과정과 근거에 실제 refs 법령명·조문이 들어갔는가
* refs 밖 수치·요건을 추가하지 않았는가
* 금지어를 쓰지 않았는가
* 근거 끝 disclaimer가 label과 맞게 붙었는가
* JSON 배열 외 다른 텍스트를 출력하지 않았는가

[입력 골든 케이스]
{골든 레코드 JSON 1건}
"""


In [16]:
import glob, json

# ── 조단위 법령 DB ──
arts = []
for f in glob.glob(JO_GLOB):                       # *_jo.json 12개
    arts.extend(json.load(open(f, encoding="utf-8"))["articles"])

LK = {a["chunk_id"]: a for a in arts}              # chunk_id → 조문 사전
ALL_IDS = list(LK.keys())                          # 전체 조문 id

def art_name(a):                                   # "지방계약법 제30조(수의계약)" 형태
    jo = (a.get("hierarchy") or {}).get("조") or a.get("article_number","")
    return f"{a['law_name']} {jo}({a.get('title','')})".strip()

print(f"법령 DB 로드 완료: {len(LK)}개 조문 (파일 {len(glob.glob(JO_GLOB))}개)")

법령 DB 로드 완료: 1186개 조문 (파일 12개)


In [11]:
# ── 골든 로드 ──
raw = json.load(open(GOLDEN, encoding="utf-8"))

# GOLDEN 파일이 {"data": [...]} 형태면 data를 쓰고,
# 그냥 [{...}, {...}] 배열 형태면 그대로 사용
if isinstance(raw, dict) and "data" in raw:
    golden = raw["data"]
elif isinstance(raw, list):
    golden = raw
else:
    raise ValueError("GOLDEN JSON 구조가 예상과 다릅니다. list 또는 {'data': [...]} 형태여야 합니다.")

def slim(rec):   # GPT에 chunk_text까지 전달 (본문 800자로 제한)
    r = dict(rec)
    r["refs"] = [
        {
            "chunk_id": x["chunk_id"],
            "article": x["article"],
            "chunk_text": x.get("chunk_text", "")[:800]   # ← 본문 포함
        }
        for x in rec["refs"]
    ]
    return r

def parse_json(t):
    t = re.sub(r"^```(json)?|```$", "", t.strip(), flags=re.M).strip()
    return json.loads(t)

In [12]:
# ── GPT-4o 증강 호출 ──
def augment(rec):
    prompt = CON_AUG_PROMPT.replace("{N}", str(N_PER_CALL)) + "\n\n[입력 골든 케이스]\n" + json.dumps(slim(rec), ensure_ascii=False)
    resp = client.chat.completions.create(
        model=MODEL, temperature=TEMP,
        messages=[{"role": "user", "content": prompt}],
    )
    return parse_json(resp.choices[0].message.content)


In [17]:
# ── 후처리: 본문 주입 · distractor · 검수 ──
BAN = ["위반","부당","무효","적법","위법","소지","정당"]
def inject_text(rec):
    for rf in rec["refs"]:
        a = LK[rf["chunk_id"]]
        rf["article"]    = art_name(a)
        rf["law_name"]   = a["law_name"]
        rf["chunk_text"] = a["text"]        # ← [:800] 제거! 전체
    return rec
def add_distractor(rec, n=1):
    if rec["label"] == "검토": return rec
    have = {rf["chunk_id"] for rf in rec["refs"]}
    for did in random.sample([i for i in ALL_IDS if i not in have], n):
        a = LK[did]
        rec["refs"].append({"chunk_id":did,"article":art_name(a),"law_name":a["law_name"],"chunk_text":a["text"][:800]})
    random.shuffle(rec["refs"]); return rec
def qc(rec):
    if rec.get("label") not in ("일치","불일치","검토"): return False, "label"
    if not all(rf.get("chunk_id") in LK for rf in rec.get("refs",[])): return False, "잘못된_id"
    txt = rec.get("사고과정","") + rec.get("근거","")
    if any(b in txt for b in BAN): return False, "금지어"
    if "법적 판단 아님" not in rec.get("근거",""): return False, "disclaimer"
    return True, ""

In [14]:
# ── 실행 (라벨별 목표 채우기 + 초과생성) ──
import math
from collections import defaultdict

random.seed(42)

# 시드를 라벨별로 그룹핑
seeds_by_label = defaultdict(list)
for rec in golden:
    seeds_by_label[rec["label"]].append(rec)

aug, dropped = [], []

for label, target in CON_TARGET.items():
    pool = seeds_by_label.get(label, [])
    if not pool:
        print(f"{label} 시드 0개, 건너뜀"); continue

    passed = []
    max_gen = int(target * OVERGEN.get(label, 2.0))   # 초과생성 상한
    gen_count = 0
    i = 0
    pbar = tqdm(total=target, desc=f"{label}")
    while len(passed) < target and gen_count < max_gen:
        seed = pool[i % len(pool)]      # 시드 순환
        i += 1
        try:
            variants = augment(seed)     # N_PER_CALL개 생성
        except Exception as e:
            dropped.append({"id": seed.get("id"), "err": str(e)}); continue
        for v in variants:
            gen_count += 1
            ok, why = qc(v)
            if not ok:
                dropped.append({"id": v.get("id"), "reason": why}); continue
            inject_text(v)
            if random.random() < DISTRACTOR_RATE:
                add_distractor(v, 1)
            passed.append(v)
            pbar.update(1)
            if len(passed) >= target:
                break
    pbar.close()
    aug.extend(passed[:target])
    print(f"{label}: 목표 {target} → 채택 {min(len(passed),target)} (생성시도 {gen_count})")

검토: 100%|██████████| 28/28 [03:53<00:00,  8.34s/it]


검토: 목표 28 → 채택 28 (생성시도 36)


불일치: 100%|██████████| 14/14 [02:15<00:00,  9.67s/it]


불일치: 목표 14 → 채택 14 (생성시도 14)


일치: 100%|██████████| 8/8 [01:01<00:00,  7.73s/it]

일치: 목표 8 → 채택 8 (생성시도 8)


In [15]:
# ── 저장 ──
# 1) 증강본만 따로 (IAA 평가·QC용 — 시드 섞이면 안 되는 경우)
json.dump(aug, open("CON_AUG_ONLY.json", "w"), ensure_ascii=False, indent=2)

# 2) 골든+증강 합본 (학습 데이터용)
full = golden + aug
json.dump(full, open("CON_AUGMENTED.json", "w"), ensure_ascii=False, indent=2)

print(f"증강본만: {len(aug)}건 → CON_AUG_ONLY.json")
print(f"골든+증강: {len(golden)} + {len(aug)} = {len(full)}건 → CON_AUGMENTED.json")
print(f"탈락: {len(dropped)}건")
if dropped: print("탈락 샘플:", dropped[:5])

증강본만: 50건 → CON_AUG_ONLY.json
골든+증강: 35 + 50 = 85건 → CON_AUGMENTED.json
탈락: 8건
탈락 샘플: [{'id': 'CON_031_aug1', 'reason': '금지어'}, {'id': 'CON_031_aug2', 'reason': '금지어'}, {'id': 'CON_031_aug3', 'reason': '금지어'}, {'id': 'CON_031_aug4', 'reason': '금지어'}, {'id': 'CON_030_aug1', 'reason': '금지어'}]


In [18]:
# 특정 JSON 파일에 조문을 채워넣는 코드
import json, glob, random

# 1) 법령 DB 로드
arts = []
for f in glob.glob("*_jo.json"):
    arts.extend(json.load(open(f, encoding="utf-8"))["articles"])
LK = {a["chunk_id"]: a for a in arts}

def art_name(a):
    jo = (a.get("hierarchy") or {}).get("조") or a.get("article_number","")
    return f"{a['law_name']} {jo}({a.get('title','')})".strip()

# 2) 채워넣을 파일 로드
FILE = "CON_AUGMENTED.json"       # ← 채우려는 파일명
data = json.load(open(FILE, encoding="utf-8"))
if isinstance(data, dict) and "data" in data:
    data = data["data"]

# 3) refs의 chunk_text 채우기
filled, missing = 0, []
for r in data:
    for rf in r.get("refs", []):
        cid = rf.get("chunk_id")
        if cid in LK:
            a = LK[cid]
            rf["article"]    = art_name(a)
            rf["law_name"]   = a["law_name"]
            rf["chunk_text"] = a["text"]      # ← 전문 주입
            filled += 1
        else:
            missing.append(cid)

# 4) 저장 (덮어쓰기)
json.dump(data, open(FILE, "w", encoding="utf-8"), ensure_ascii=False, indent=2)
print(f"채움 {filled}개 refs / LK에 없는 id {len(missing)}개")
if missing: print("없는 id 예시:", list(set(missing))[:10])

채움 145개 refs / LK에 없는 id 0개
